In [1]:
# Building the Master Dataset
# merging all 5 clean datasets into one master csv for dashboard and ml models

import pandas as pd

# loading all clean datasets
df_nomis = pd.read_csv('NOMIS_Vacancies_Clean.csv')
df_ashe = pd.read_csv('ASHE_Clean.csv')
df_immig = pd.read_csv('HO_ImmigStats_Clean.csv')

print("NOMIS shape:", df_nomis.shape)
print("ASHE shape:", df_ashe.shape)
print("ImmigStats shape:", df_immig.shape)

print("\nNOMIS columns:", df_nomis.columns.tolist())
print("ASHE columns:", df_ashe.columns.tolist())
print("ImmigStats columns:", df_immig.columns.tolist())

NOMIS shape: (20, 6)
ASHE shape: (25, 3)
ImmigStats shape: (30, 3)

NOMIS columns: ['Quarter', 'Technology', 'Healthcare', 'Finance', 'Engineering', 'Education']
ASHE columns: ['Year', 'Sector', 'Median_Salary']
ImmigStats columns: ['Year', 'Sector', 'Visa_Grants']


In [2]:
# NOMIS is in wide format - need to convert to long format to merge
df_nomis_long = df_nomis.melt(id_vars=['Quarter'], 
                               var_name='Sector', 
                               value_name='Vacancy_Count')

# extracting year from quarter for merging with ASHE and ImmigStats
df_nomis_long['Year'] = df_nomis_long['Quarter'].apply(lambda x: int(x.split('-')[0]))

print("NOMIS long format shape:", df_nomis_long.shape)
print("\nfirst 5 rows:")
print(df_nomis_long.head())

NOMIS long format shape: (100, 4)

first 5 rows:
   Quarter      Sector  Vacancy_Count  Year
0  2021-Q1  Technology        1514194  2021
1  2021-Q2  Technology        1497237  2021
2  2021-Q3  Technology        1534787  2021
3  2021-Q4  Technology        1564133  2021
4  2022-Q1  Technology        1573034  2022


In [3]:
# merging NOMIS with ASHE on Year + Sector
df_merge1 = pd.merge(df_nomis_long, df_ashe, on=['Year', 'Sector'], how='left')
print("after merging NOMIS + ASHE:", df_merge1.shape)
print("\nmissing values:")
print(df_merge1.isnull().sum())

after merging NOMIS + ASHE: (100, 5)

missing values:
Quarter          0
Sector           0
Vacancy_Count    0
Year             0
Median_Salary    0
dtype: int64


In [4]:
# merging with ImmigStats on Year + Sector
df_master = pd.merge(df_merge1, df_immig, on=['Year', 'Sector'], how='left')
print("after merging all 3:", df_master.shape)
print("\nmissing values:")
print(df_master.isnull().sum())
print("\nfirst 10 rows:")
print(df_master.head(10))

after merging all 3: (100, 6)

missing values:
Quarter          0
Sector           0
Vacancy_Count    0
Year             0
Median_Salary    0
Visa_Grants      0
dtype: int64

first 10 rows:
   Quarter      Sector  Vacancy_Count  Year  Median_Salary  Visa_Grants
0  2021-Q1  Technology        1514194  2021          40826        14328
1  2021-Q2  Technology        1497237  2021          40826        14328
2  2021-Q3  Technology        1534787  2021          40826        14328
3  2021-Q4  Technology        1564133  2021          40826        14328
4  2022-Q1  Technology        1573034  2022          43005        22529
5  2022-Q2  Technology        1610644  2022          43005        22529
6  2022-Q3  Technology        1625649  2022          43005        22529
7  2022-Q4  Technology        1622771  2022          43005        22529
8  2023-Q1  Technology        1647451  2023          46042        15484
9  2023-Q2  Technology        1643111  2023          46042        15484


In [5]:
# reordering columns nicely
df_master = df_master[['Quarter', 'Year', 'Sector', 'Vacancy_Count', 
                        'Median_Salary', 'Visa_Grants']]

# quick validation
print("final master dataset:")
print("- shape:", df_master.shape)
print("- sectors:", df_master['Sector'].unique().tolist())
print("- quarters:", df_master['Quarter'].unique().tolist())
print("- zero nulls:", df_master.isnull().sum().sum() == 0)

final master dataset:
- shape: (100, 6)
- sectors: ['Technology', 'Healthcare', 'Finance', 'Engineering', 'Education']
- quarters: ['2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4', '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4', '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']
- zero nulls: True


In [6]:
# saving master dataset
df_master.to_csv('Master_Dataset_v1.csv', index=False)
print("saved! Master_Dataset_v1.csv")
print("\nfull preview:")
print(df_master.to_string())

saved! Master_Dataset_v1.csv

full preview:
    Quarter  Year       Sector  Vacancy_Count  Median_Salary  Visa_Grants
0   2021-Q1  2021   Technology        1514194          40826        14328
1   2021-Q2  2021   Technology        1497237          40826        14328
2   2021-Q3  2021   Technology        1534787          40826        14328
3   2021-Q4  2021   Technology        1564133          40826        14328
4   2022-Q1  2022   Technology        1573034          43005        22529
5   2022-Q2  2022   Technology        1610644          43005        22529
6   2022-Q3  2022   Technology        1625649          43005        22529
7   2022-Q4  2022   Technology        1622771          43005        22529
8   2023-Q1  2023   Technology        1647451          46042        15484
9   2023-Q2  2023   Technology        1643111          46042        15484
10  2023-Q3  2023   Technology        1584917          46042        15484
11  2023-Q4  2023   Technology        1628286          46042        